In [193]:
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import ARLSTem
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [194]:
df_train = pd.read_csv('/content/nlp_train.csv')
df_test = pd.read_csv('/content/nlp_test.csv')

In [195]:
df_train.head()

,text,FinalCB
0,من رحم المعاناه السوري عبد المولي مبتور الساق...,no
1,سناب عين المشاهد سناب اخباري متنوع امطار حواد...,no
2,بعض العرب ودولها اشد خطرا من اليهود انفسهم ول...,yes
3,🇮 🇷 الفارسيه عين العروبه 💛 🖤,no
4,حزب اله هو المسيطر علي كل مفاصل واجهزه الدوله...,no


In [196]:
df_test.head()

,text,FinalCB
0,حياك اله اخي عبد العزيز وعظم اله اجر الامه في...,no
1,رحم اله البطل هواري بومدين والذي قال نحن مع ف...,no
2,يلي قتل ضباط و عناصر الجيش ارهابي ولازم ينعدم...,no
3,عبد الرحمن رافع العمري جندي سعودي استشهد عام ...,no
4,تزايد حالات الاصابه بفيروس يدفع الحكومه البري...,no


In [197]:
df_train['FinalCB'].replace({'yes': 1, 'no': 0}, inplace=True)
df_train.head()

,text,FinalCB
0,من رحم المعاناه السوري عبد المولي مبتور الساق...,0
1,سناب عين المشاهد سناب اخباري متنوع امطار حواد...,0
2,بعض العرب ودولها اشد خطرا من اليهود انفسهم ول...,1
3,🇮 🇷 الفارسيه عين العروبه 💛 🖤,0
4,حزب اله هو المسيطر علي كل مفاصل واجهزه الدوله...,0


In [198]:
df_test['FinalCB'].replace({'yes': 1, 'no': 0}, inplace=True)
df_test.head()

,text,FinalCB
0,حياك اله اخي عبد العزيز وعظم اله اجر الامه في...,0
1,رحم اله البطل هواري بومدين والذي قال نحن مع ف...,0
2,يلي قتل ضباط و عناصر الجيش ارهابي ولازم ينعدم...,0
3,عبد الرحمن رافع العمري جندي سعودي استشهد عام ...,0
4,تزايد حالات الاصابه بفيروس يدفع الحكومه البري...,0


In [199]:
df_train.dtypes

 text      object
FinalCB     int64
dtype: object

In [200]:
def remove_emoji(text):
  emoji_pattern = re.compile("["
  u"\U0001F600-\U0001F64F" # emoticons
  u"\U0001F300-\U0001F5FF" # symbols & pictographs
  u"\U0001F680-\U0001F6FF" # transport & map symbols
  u"\U0001F1E0-\U0001F1FF" # flags (iOS)
  u"\U00002702-\U000027B0"
  u"\U000024C2-\U0001F251"
  "]+", flags=re.UNICODE)
  text = emoji_pattern.sub(r'', text)
  return text

In [201]:
def clean_text(tweet):
  text = re.sub(":","", tweet)
  text = re.sub("\d+", "", text)
  text = re.sub("\.+", "", text)
  text = remove_emoji(text)
  return text

In [202]:
df_train[' text'] = df_train[' text'].apply(clean_text)
df_test[' text'] = df_test[' text'].apply(clean_text)

In [203]:
stp = stopwords.words('arabic')
stp

['إذ',
 'إذا',
 'إذما',
 'إذن',
 'أف',
 'أقل',
 'أكثر',
 'ألا',
 'إلا',
 'التي',
 'الذي',
 'الذين',
 'اللاتي',
 'اللائي',
 'اللتان',
 'اللتيا',
 'اللتين',
 'اللذان',
 'اللذين',
 'اللواتي',
 'إلى',
 'إليك',
 'إليكم',
 'إليكما',
 'إليكن',
 'أم',
 'أما',
 'أما',
 'إما',
 'أن',
 'إن',
 'إنا',
 'أنا',
 'أنت',
 'أنتم',
 'أنتما',
 'أنتن',
 'إنما',
 'إنه',
 'أنى',
 'أنى',
 'آه',
 'آها',
 'أو',
 'أولاء',
 'أولئك',
 'أوه',
 'آي',
 'أي',
 'أيها',
 'إي',
 'أين',
 'أين',
 'أينما',
 'إيه',
 'بخ',
 'بس',
 'بعد',
 'بعض',
 'بك',
 'بكم',
 'بكم',
 'بكما',
 'بكن',
 'بل',
 'بلى',
 'بما',
 'بماذا',
 'بمن',
 'بنا',
 'به',
 'بها',
 'بهم',
 'بهما',
 'بهن',
 'بي',
 'بين',
 'بيد',
 'تلك',
 'تلكم',
 'تلكما',
 'ته',
 'تي',
 'تين',
 'تينك',
 'ثم',
 'ثمة',
 'حاشا',
 'حبذا',
 'حتى',
 'حيث',
 'حيثما',
 'حين',
 'خلا',
 'دون',
 'ذا',
 'ذات',
 'ذاك',
 'ذان',
 'ذانك',
 'ذلك',
 'ذلكم',
 'ذلكما',
 'ذلكن',
 'ذه',
 'ذو',
 'ذوا',
 'ذواتا',
 'ذواتي',
 'ذي',
 'ذين',
 'ذينك',
 'ريث',
 'سوف',
 'سوى',
 'شتان',
 'عدا',
 'عسى',
 'عل'

In [204]:
X_train = df_train[' text'].to_list()
y_train= df_train['FinalCB']

X_test = df_test[' text'].to_list()
y_test = df_test['FinalCB']

In [205]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
feature_names = vectorizer.get_feature_names_out()

In [206]:
df1 = pd.DataFrame(X_train.toarray(), columns = feature_names)
df2 = pd.DataFrame(X_test.toarray(), columns = feature_names)

In [207]:
from sklearn.metrics import f1_score

In [208]:
lr = LogisticRegression()
lr.fit(df1, df_train['FinalCB'])
prd = lr.predict(df2)
print(accuracy_score(df_test['FinalCB'] , prd ))

0.8357380688124306


In [209]:
f1 = f1_score(y_test, prd)

print("F1 score:", f1)

F1 score: 0.7764350453172206


In [210]:
submission_df = pd.DataFrame({'Id': df_test[' text'].index + 1, 'FinalCB': prd})

In [211]:
submission_df.head()

,Id,FinalCB
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [212]:
submission_df.to_csv("lana2.csv", index=False)